# Daily and weekly sales distributions

This notebook gives a compact overview of the processed demand series at their native aggregation levels. Each aggregation is analysed for both the regular dataset and the FCM dataset.

The unit of analysis is an `ARTIKEL_ID` x `MARKT_ID` series. A period is active when a row exists; a demand period has positive `ABVERKAUFTE_MENGE_KG`. The weekly data additionally contains the number of active, demand, and zero-sales days in each week.


In [ ]:
from pathlib import Path
from io import BytesIO
import os
os.environ.setdefault("MPLCONFIGDIR", "/tmp/matplotlib")
Path(os.environ["MPLCONFIGDIR"]).mkdir(parents=True, exist_ok=True)
import matplotlib
matplotlib.use("Agg")
import duckdb
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from IPython.display import display, Image
pd.set_option("display.max_columns", 30)
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (11, 5)
GROUP_COLS = ["ARTIKEL_ID", "MARKT_ID"]
DEMAND_COL = "ABVERKAUFTE_MENGE_KG"
DATA_ROOT = next((p for p in [Path("../../data/processed"), Path("../data/processed"), Path("data/processed")] if p.exists()), None)
if DATA_ROOT is None: raise FileNotFoundError("Could not find data/processed from the notebook directory.")
DATASETS = {"daily": {"regular": DATA_ROOT / "transactions_dst_over_days", "fcm": DATA_ROOT / "transactions_dst_over_days_fcm"}, "weekly": {"regular": DATA_ROOT / "transactions_dst_over_weeks", "fcm": DATA_ROOT / "transactions_dst_over_weeks_fcm"}}
for aggregation in DATASETS:
    for variant, path in DATASETS[aggregation].items():
        if not list(path.glob("*.parquet")): raise FileNotFoundError(f"No parquet files found in {path}")
con = duckdb.connect()
con.execute("PRAGMA threads=4")


## Analysis helpers

The summaries below are generated with one function so daily and weekly results use the same definitions. The weekly `demand_day_share` is the share of active open days with positive sales, while `demand_period_share` is the share of active weeks with positive weekly demand.


In [ ]:
def load_series_metrics(aggregation, variant, data_dir):
    path = str(data_dir / "*.parquet")
    if aggregation == "daily":
        query = f"""
        WITH rows AS (
            SELECT ARTIKEL_ID, MARKT_ID, CAST(COALESCE({DEMAND_COL}, 0) AS DOUBLE) AS demand, CAST(DATE AS DATE) AS period
            FROM read_parquet(?)
        )
        SELECT ARTIKEL_ID, MARKT_ID, COUNT(*)::INTEGER AS active_periods,
               SUM((demand > 0)::INTEGER)::INTEGER AS demand_periods,
               SUM((demand <= 0)::INTEGER)::INTEGER AS zero_periods,
               SUM(demand) AS total_demand, MIN(period) AS first_period, MAX(period) AS last_period,
               NULL::DOUBLE AS demand_day_share
        FROM rows GROUP BY ARTIKEL_ID, MARKT_ID
        """
    else:
        query = f"""
        SELECT ARTIKEL_ID, MARKT_ID, COUNT(*)::INTEGER AS active_periods,
               SUM((COALESCE({DEMAND_COL}, 0) > 0)::INTEGER)::INTEGER AS demand_periods,
               SUM((COALESCE({DEMAND_COL}, 0) <= 0)::INTEGER)::INTEGER AS zero_periods,
               SUM(COALESCE({DEMAND_COL}, 0)) AS total_demand,
               MIN(CAST(DATE AS DATE)) AS first_period, MAX(CAST(DATE AS DATE)) AS last_period,
               SUM(COALESCE(demand_days_in_week, 0)) / NULLIF(SUM(COALESCE(active_days_in_week, 0)), 0) AS demand_day_share
        FROM read_parquet(?) GROUP BY ARTIKEL_ID, MARKT_ID
        """
    result = con.execute(query, [path]).fetchdf()
    result["aggregation"], result["variant"] = aggregation, variant
    result["demand_period_share"] = result["demand_periods"] / result["active_periods"]
    if aggregation == "daily": result["demand_day_share"] = result["demand_period_share"]
    result["zero_period_share"] = result["zero_periods"] / result["active_periods"]
    result["dataset"] = aggregation.title() + " / " + variant.title()
    return result

metrics = pd.concat([load_series_metrics(a, v, p) for a, variants in DATASETS.items() for v, p in variants.items()], ignore_index=True)
DATASET_ORDER = ["Daily / Regular", "Daily / Fcm", "Weekly / Regular", "Weekly / Fcm"]
metrics["dataset"] = pd.Categorical(metrics["dataset"], categories=DATASET_ORDER, ordered=True)
metrics.head()


# Daily analysis

Daily rows represent the active open-day periods created by the data preparation pipeline.


In [ ]:
daily = metrics[metrics["aggregation"] == "daily"].copy()
daily_overview = (daily.groupby("dataset", observed=True).agg(series=("ARTIKEL_ID", "size"), stores=("MARKT_ID", "nunique"), products=("ARTIKEL_ID", "nunique"), active_days=("active_periods", "sum"), demand_days=("demand_periods", "sum"), zero_days=("zero_periods", "sum"), total_demand_kg=("total_demand", "sum")).reset_index())
daily_overview["demand_day_share"] = daily_overview["demand_days"] / daily_overview["active_days"]
display(daily_overview.style.format({"total_demand_kg":"{:,.1f}", "demand_day_share":"{:.1%}"}))


In [ ]:
def distribution_table(frame, columns):
    rows = []
    for dataset, group in frame.groupby("dataset", observed=True):
        for column in columns:
            q = group[column].dropna().quantile([.25, .5, .75, .95])
            rows.append({"dataset":dataset, "measure":column, "p25":q.loc[.25], "median":q.loc[.5], "p75":q.loc[.75], "p95":q.loc[.95]})
    return pd.DataFrame(rows)

quantile_format = {"p25":"{:.2f}", "median":"{:.2f}", "p75":"{:.2f}", "p95":"{:.2f}"}
display(distribution_table(daily, ["active_periods", "demand_periods", "zero_periods", "demand_day_share"]).style.format(quantile_format))
daily_plot = daily.copy()
daily_plot["Datensatz"] = daily_plot["dataset"].astype(str).replace({"Daily / Regular": "Täglich / Regulär", "Daily / Fcm": "Täglich / FCM"})
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sns.histplot(data=daily_plot, x="active_periods", hue="Datensatz", bins=30, element="step", stat="density", common_norm=False, ax=axes[0])
sns.histplot(data=daily_plot, x="demand_periods", hue="Datensatz", bins=30, element="step", stat="density", common_norm=False, ax=axes[1])
axes[0].set(title="Aktive Tage je Zeitreihe", xlabel="Anzahl aktiver Tage", ylabel="Dichte")
axes[1].set(title="Nachfragetage je Zeitreihe", xlabel="Anzahl der Nachfragetage", ylabel="Dichte")
plt.tight_layout()
figure_buffer = BytesIO()
fig.savefig(figure_buffer, format="png", dpi=120, bbox_inches="tight")
display(Image(data=figure_buffer.getvalue()))
plt.close(fig)


## Daily binned count distributions

Bins make the long-tailed period counts easier to compare than raw histograms.


In [ ]:
def binned_counts(frame, columns, bins, labels):
    parts = []
    for column in columns:
        part = frame[["dataset", column]].copy(); part["measure"] = column
        part["bin"] = pd.cut(part[column], bins=bins, labels=labels, right=True)
        counts = part.groupby(["dataset", "measure", "bin"], observed=False).size().rename("series").reset_index()
        counts["share"] = counts["series"] / counts.groupby(["dataset", "measure"], observed=False)["series"].transform("sum")
        parts.append(counts)
    return pd.concat(parts, ignore_index=True)
DAY_BINS = [-1, 0, 1, 7, 14, 30, 60, 90, 180, 365, 730, np.inf]
DAY_LABELS = ["0", "1", "2-7", "8-14", "15-30", "31-60", "61-90", "91-180", "181-365", "366-730", ">730"]
daily_bins = binned_counts(daily, ["active_periods", "demand_periods"], DAY_BINS, DAY_LABELS)
display(daily_bins.pivot_table(index=["measure", "bin"], columns="dataset", values="share", observed=False).style.format("{:.1%}"))


The `top_store_products` table summarizes where demand is concentrated. It first sums demand for each store-product pair (`MARKT_ID`, `ARTIKEL_ID`), then ranks stores by the number of products with positive demand and by total demand. `top_product_ids` lists the three product IDs with the highest total demand in that store. The `top_store_products` table summarizes where demand is concentrated.


In [ ]:
def top_store_products(frame, n=10):
    store_product = frame.groupby(["MARKT_ID", "ARTIKEL_ID"], observed=True).agg(demand=("total_demand", "sum")).reset_index()
    top_stores = (store_product.groupby("MARKT_ID", as_index=False).agg(products_with_demand=("ARTIKEL_ID", "nunique"), total_demand=("demand", "sum")).sort_values(["products_with_demand", "total_demand"], ascending=False).head(n))
    top_products = store_product.sort_values(["MARKT_ID", "demand"], ascending=[True, False]).groupby("MARKT_ID", as_index=False).head(3)
    ids = (top_products.assign(product_id=top_products["ARTIKEL_ID"].astype(str)).groupby("MARKT_ID")["product_id"].agg(lambda values: ", ".join(values)).rename("top_product_ids"))
    return top_stores.merge(ids, on="MARKT_ID", how="left")
for dataset, group in daily.groupby("dataset", observed=True):
    print(dataset); display(top_store_products(group))


## Daily FCM: Muster zur Wahl des ersten Trainingsfensters

Diese Voranalyse sucht nach einem sinnvollen Startwert für `START_MIN_TRAIN_SIZE`, bevor die eigentliche Filterprüfung beginnt. Für mehrere Fenstergrößen wird geprüft, wie viele Reihen mindestens fünf Nachfragetage enthalten. Das Wochentagsprofil im ersten Fenster wird ausschließlich mit Beobachtungen nach diesem Fenster verglichen; die späteren Beobachtungen müssen mindestens vier aktive Verkaufswochen umfassen. Als datengetriebener Kandidat gilt das kleinste Fenster, bei dem mindestens 50 % der Reihen schätzbar sind und die mediane Korrelation mit den späteren Wochentagsprofilen mindestens 0,70 beträgt. Erfüllt kein Fenster beide Kriterien, bleibt 121 aktive Tage der konservative Startwert für den anschließenden Test. Zusätzlich zeigen drei Reihen mit niedrigem, mittlerem und hohem Nachfragetageanteil ihre gesamte aktive Historie. Die geglättete Linie entspricht ungefähr einer aktiven Verkaufswoche.


In [ ]:
PATTERN_MIN_DEMAND_PERIODS = 5
PATTERN_ESTIMABLE_SHARE = 0.50
PATTERN_CORRELATION_THRESHOLD = 0.70
PATTERN_FALLBACK_START_MIN_TRAIN_SIZE = 121
PATTERN_WINDOW_CANDIDATES = [31, 61, 91, 121, 151, 181]

daily_fcm_periods = con.execute(
    f"""
    SELECT
        ARTIKEL_ID,
        MARKT_ID,
        CAST(DATE AS DATE) AS period,
        SUM(CAST(COALESCE({DEMAND_COL}, 0) AS DOUBLE)) AS demand
    FROM read_parquet(?)
    GROUP BY ARTIKEL_ID, MARKT_ID, period
    ORDER BY ARTIKEL_ID, MARKT_ID, period
    """,
    [str(DATASETS["daily"]["fcm"] / "*.parquet")],
).fetchdf()
daily_fcm_periods["active_day_number"] = (
    daily_fcm_periods.groupby(GROUP_COLS, observed=True).cumcount() + 1
)
daily_fcm_periods["weekday_number"] = pd.to_datetime(
    daily_fcm_periods["period"]
).dt.weekday
active_weekdays = sorted(daily_fcm_periods["weekday_number"].unique())
active_days_per_week = len(active_weekdays)
total_pattern_series = daily_fcm_periods.groupby(GROUP_COLS, observed=True).ngroups

min_later_active_days = active_days_per_week * 4
pattern_rows = []
for candidate in PATTERN_WINDOW_CANDIDATES:
    candidate_data = daily_fcm_periods[
        daily_fcm_periods["active_day_number"] <= candidate
    ]
    later_data = daily_fcm_periods[
        daily_fcm_periods["active_day_number"] > candidate
    ]
    candidate_metrics = (
        candidate_data.groupby(GROUP_COLS, observed=True)
        .agg(
            active_days=("period", "size"),
            demand_days=("demand", lambda demand: (demand > 0).sum()),
        )
    )
    estimable_keys = candidate_metrics[
        (candidate_metrics["active_days"] == candidate)
        & (candidate_metrics["demand_days"] >= PATTERN_MIN_DEMAND_PERIODS)
    ].index
    later_metrics = later_data.groupby(GROUP_COLS, observed=True).agg(
        later_active_days=("period", "size")
    )
    comparable_keys = [
        series_key for series_key in estimable_keys
        if series_key in later_metrics.index
        and later_metrics.loc[series_key, "later_active_days"] >= min_later_active_days
    ]
    candidate_profiles = (
        candidate_data.groupby([*GROUP_COLS, "weekday_number"], observed=True)["demand"]
        .mean()
        .unstack(fill_value=0)
        .reindex(columns=active_weekdays, fill_value=0)
    )
    later_profiles = (
        later_data.groupby([*GROUP_COLS, "weekday_number"], observed=True)["demand"]
        .mean()
        .unstack(fill_value=0)
        .reindex(columns=active_weekdays, fill_value=0)
    )
    profile_correlations = []
    for series_key in comparable_keys:
        candidate_profile = candidate_profiles.loc[series_key].to_numpy(dtype=float)
        later_profile = later_profiles.loc[series_key].to_numpy(dtype=float)
        if candidate_profile.std() > 0 and later_profile.std() > 0:
            profile_correlations.append(np.corrcoef(candidate_profile, later_profile)[0, 1])
    estimable_series = len(estimable_keys)
    pattern_rows.append({
        "aktive_tage": candidate,
        "schaetzbare_reihen": estimable_series,
        "reihen_mit_spaeterem_profil": len(comparable_keys),
        "anteil_schaetzbar": estimable_series / total_pattern_series,
        "mediane_korrelation_mit_spaeter": (
            float(np.nanmedian(profile_correlations)) if profile_correlations else np.nan
        ),
    })

pattern_window_results = pd.DataFrame(pattern_rows)
display(pattern_window_results.style.format({
    "anteil_schaetzbar": "{:.1%}",
    "mediane_korrelation_mit_spaeter": "{:.2f}",
}))
pattern_candidates = pattern_window_results[
    (pattern_window_results["anteil_schaetzbar"] >= PATTERN_ESTIMABLE_SHARE)
    & (pattern_window_results["mediane_korrelation_mit_spaeter"] >= PATTERN_CORRELATION_THRESHOLD)
]
if pattern_candidates.empty:
    suggested_start_min_train_size = PATTERN_FALLBACK_START_MIN_TRAIN_SIZE
    print("Kein Kandidat erfüllt beide Kriterien; 121 aktive Tage bleiben der konservative Startwert.")
else:
    suggested_start_min_train_size = int(pattern_candidates.iloc[0]["aktive_tage"])
print(f"Vorgeschlagener START_MIN_TRAIN_SIZE: {suggested_start_min_train_size} aktive Tage.")

full_series_summary = (
    daily_fcm_periods.groupby(GROUP_COLS, observed=True)
    .agg(
        active_days=("period", "size"),
        demand_days=("demand", lambda demand: (demand > 0).sum()),
    )
    .reset_index()
)
full_series_summary = full_series_summary[
    full_series_summary["demand_days"] >= PATTERN_MIN_DEMAND_PERIODS
].copy()
full_series_summary["demand_day_share"] = (
    full_series_summary["demand_days"] / full_series_summary["active_days"]
)
full_series_summary = full_series_summary.sort_values("demand_day_share").reset_index(drop=True)
pattern_positions = np.linspace(0, len(full_series_summary) - 1, min(3, len(full_series_summary)), dtype=int)
pattern_examples = full_series_summary.iloc[pattern_positions].copy()
pattern_product_names = con.execute(
    """
    SELECT ARTIKEL_ID, ANY_VALUE(ARTIKEL_BEZ) AS ARTIKEL_BEZ
    FROM read_parquet(?)
    GROUP BY ARTIKEL_ID
    """,
    [str(DATASETS["daily"]["fcm"] / "*.parquet")],
).fetchdf()
pattern_examples = pattern_examples.merge(pattern_product_names, on="ARTIKEL_ID", how="left")

fig, axes = plt.subplots(len(pattern_examples), 1, figsize=(14, 3.2 * len(pattern_examples)), squeeze=False)
for row, product in pattern_examples.reset_index(drop=True).iterrows():
    series = daily_fcm_periods[
        (daily_fcm_periods["ARTIKEL_ID"] == product["ARTIKEL_ID"])
        & (daily_fcm_periods["MARKT_ID"] == product["MARKT_ID"])
    ].sort_values("period").copy()
    series["wochenmittel"] = series["demand"].rolling(
        active_days_per_week, min_periods=1
    ).mean()
    ax = axes[row, 0]
    ax.plot(series["period"], series["demand"], color="#9ca3a6", linewidth=0.8, label="Tagesnachfrage")
    ax.plot(series["period"], series["wochenmittel"], color="#315b63", linewidth=1.8, label=f"Mittel über {active_days_per_week} aktive Tage")
    product_name = product["ARTIKEL_BEZ"] if pd.notna(product["ARTIKEL_BEZ"]) else "Unbekanntes Produkt"
    ax.set_title(
        f"{product_name}: vollständige aktive Historie\n"
        f"{int(product['demand_days'])} von {int(product['active_days'])} Tagen mit Nachfrage"
    )
    ax.set_ylabel("Nachfrage (kg)")
    ax.tick_params(axis="x", labelrotation=45)
    plt.setp(ax.get_xticklabels(), ha="right")
    ax.legend(loc="upper right", frameon=False)
fig.suptitle("Tägliche FCM-Reihen über die vollständige aktive Historie", fontsize=14, y=1.01)
fig.tight_layout()
figure_buffer = BytesIO()
fig.savefig(figure_buffer, format="png", dpi=120, bbox_inches="tight")
display(Image(data=figure_buffer.getvalue()))
plt.close(fig)


## Daily FCM: Selecting `DEFAULT_MIN_TRAIN_SIZE` for ADI/CV2 clustering and Modelling

This cell tests the starting window suggested by the preceding pattern analysis. It finds the smallest first training window that makes ADI and CV2 estimable for at least 50% of the daily FCM product-store series. Active days are observed rows for a series, not calendar days. A series is estimable only when it has the complete training window and at least `DEFAULT_MIN_DEMAND_PERIODS = 5` days with positive demand inside that window. If more than 50% of all series are filtered out, the window is expanded by 10 active days and tested again. The selected value is the first window that meets the 50% rule.


In [ ]:
START_MIN_TRAIN_SIZE = suggested_start_min_train_size
DEFAULT_MIN_DEMAND_PERIODS = 5
TRAIN_SIZE_STEP = 10
MAX_FILTERED_SHARE = 0.50

total_series = daily_fcm_periods.groupby(GROUP_COLS, observed=True).ngroups
max_active_days = int(daily_fcm_periods["active_day_number"].max())

def evaluate_min_train_size(min_train_size):
    first_window = daily_fcm_periods[
        daily_fcm_periods["active_day_number"] <= min_train_size
    ]
    window_metrics = (
        first_window.groupby(GROUP_COLS, observed=True)
        .agg(
            active_days_in_window=("period", "size"),
            demand_periods_in_window=("demand", lambda demand: (demand > 0).sum()),
        )
        .reset_index()
    )
    window_metrics["has_full_window"] = (
        window_metrics["active_days_in_window"] == min_train_size
    )
    window_metrics["estimable"] = (
        window_metrics["has_full_window"]
        & (window_metrics["demand_periods_in_window"] >= DEFAULT_MIN_DEMAND_PERIODS)
    )

    estimable = int(window_metrics["estimable"].sum())
    filtered = total_series - estimable
    insufficient_history = int((~window_metrics["has_full_window"]).sum())
    insufficient_demand = int(
        (
            window_metrics["has_full_window"]
            & (window_metrics["demand_periods_in_window"] < DEFAULT_MIN_DEMAND_PERIODS)
        ).sum()
    )
    return {
        "min_train_size_active_days": min_train_size,
        "total_series": total_series,
        "estimable_series": estimable,
        "filtered_series": filtered,
        "filtered_share": filtered / total_series,
        "insufficient_history": insufficient_history,
        "fewer_than_5_demand_days": insufficient_demand,
    }

window_tests = []
candidate = START_MIN_TRAIN_SIZE
while candidate <= max_active_days:
    result = evaluate_min_train_size(candidate)
    window_tests.append(result)
    if result["filtered_share"] <= MAX_FILTERED_SHARE:
        break
    candidate += TRAIN_SIZE_STEP

window_results = pd.DataFrame(window_tests)
display(window_results.style.format({"filtered_share": "{:.1%}"}))

acceptable = window_results[window_results["filtered_share"] <= MAX_FILTERED_SHARE]
if acceptable.empty:
    print(
        f"No tested window retained at least {1 - MAX_FILTERED_SHARE:.0%} of all series. "
        "The demand-period threshold or series scope must be reconsidered."
    )
else:
    selected_min_train_size = int(acceptable.iloc[0]["min_train_size_active_days"])
    selected_estimable_series = int(acceptable.iloc[0]["estimable_series"])
    print(
        f"Selected DEFAULT_MIN_TRAIN_SIZE: {selected_min_train_size} active days; "
        f"{selected_estimable_series} of {total_series} series are estimable."
    )


### Repräsentative Produkte aus bis zu zwei Filialen

Die Darstellung verwendet höchstens zwei Filialen mit jeweils drei geeigneten Produkten. Für jedes Produkt steht links die tägliche Nachfrage und rechts die mittlere Nachfrage je Wochentag. Eine Filiale ergibt damit drei Produktzeilen mit jeweils zwei Diagrammen, also sechs Diagramme. Zwei Filialen ergeben sechs Produktzeilen und insgesamt zwölf Diagramme. Pro Filiale werden Produkte mit niedrigem, mittlerem und hohem Nachfragetageanteil ausgewählt. Jedes Produkt besitzt alle `START_MIN_TRAIN_SIZE` aktiven Tage und darin mindestens fünf Tage mit positiver Nachfrage. Gepunktete Linien markieren Kalenderwochen.

In [ ]:
if acceptable.empty:
    raise RuntimeError("Representative products require an accepted training window.")

plot_window_size = START_MIN_TRAIN_SIZE
selected_window = daily_fcm_periods[
    daily_fcm_periods["active_day_number"] <= plot_window_size
].copy()
survivor_summary = (
    selected_window.groupby(GROUP_COLS, observed=True)
    .agg(
        active_days=("period", "size"),
        demand_days=("demand", lambda demand: (demand > 0).sum()),
    )
    .reset_index()
)
survivor_summary = survivor_summary[
    (survivor_summary["active_days"] == plot_window_size)
    & (survivor_summary["demand_days"] >= DEFAULT_MIN_DEMAND_PERIODS)
].copy()
survivor_summary["demand_day_share"] = (
    survivor_summary["demand_days"] / survivor_summary["active_days"]
)
store_counts = survivor_summary.groupby("MARKT_ID", observed=True).size().sort_values(ascending=False)
eligible_stores = store_counts[store_counts >= 3].head(2)
if eligible_stores.empty:
    raise RuntimeError("Es wird mindestens eine Filiale mit drei geeigneten Produkten benötigt.")
selected_store_ids = eligible_stores.index.tolist()
representative_parts = []
for store_number, store_id in enumerate(selected_store_ids, start=1):
    store_survivors = (
        survivor_summary[survivor_summary["MARKT_ID"] == store_id]
        .sort_values("demand_day_share")
        .reset_index(drop=True)
    )
    representative_positions = np.linspace(0, len(store_survivors) - 1, 3, dtype=int)
    store_representatives = store_survivors.iloc[representative_positions].copy()
    store_representatives["store_number"] = store_number
    representative_parts.append(store_representatives)
representatives = pd.concat(representative_parts, ignore_index=True)
product_names = con.execute(
    """
    SELECT ARTIKEL_ID, ANY_VALUE(ARTIKEL_BEZ) AS ARTIKEL_BEZ
    FROM read_parquet(?)
    GROUP BY ARTIKEL_ID
    """,
    [str(DATASETS["daily"]["fcm"] / "*.parquet")],
).fetchdf()
representatives = representatives.merge(product_names, on="ARTIKEL_ID", how="left")

weekday_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday"]
weekday_labels = ["Mo", "Di", "Mi", "Do", "Fr", "Sa"]
store_count = len(selected_store_ids)
product_rows = store_count * 3
fig, axes = plt.subplots(
    product_rows, 2, figsize=(14, 3.6 * product_rows), squeeze=False
)

for store_row, store_id in enumerate(selected_store_ids):
    store_products = representatives[representatives["MARKT_ID"] == store_id].reset_index(drop=True)
    for product_position, product in store_products.iterrows():
        series = selected_window[
            (selected_window["ARTIKEL_ID"] == product["ARTIKEL_ID"])
            & (selected_window["MARKT_ID"] == product["MARKT_ID"])
        ].sort_values("period").copy()
        series["weekday"] = pd.to_datetime(series["period"]).dt.day_name()
        plot_row = store_row * 3 + product_position
        ax_daily, ax_weekday = axes[plot_row]
        ax_daily.plot(series["period"], series["demand"], color="#315b63", linewidth=1.2)
        ax_daily.scatter(
            series.loc[series["demand"] > 0, "period"],
            series.loc[series["demand"] > 0, "demand"],
            color="#c4513b", s=22, zorder=3
        )
        for week_start in pd.date_range(series["period"].min(), series["period"].max(), freq="W-MON"):
            ax_daily.axvline(week_start, color="#9ca3a6", linestyle=":", linewidth=0.7, alpha=0.7)
        product_name = product["ARTIKEL_BEZ"] if pd.notna(product["ARTIKEL_BEZ"]) else "Unbekanntes Produkt"
        ax_daily.set_title(
            f"Filiale {store_row + 1}: {product_name}\n"
            f"{int(product['demand_days'])} von {plot_window_size} Tagen mit Nachfrage"
        )
        ax_daily.set_ylabel("Tagesnachfrage (kg)")
        ax_daily.tick_params(axis="x", labelrotation=45)
        plt.setp(ax_daily.get_xticklabels(), ha="right")
        ax_daily.legend(loc="upper right", frameon=False)

        weekday_mean = (
            series.groupby("weekday", observed=True)["demand"]
            .mean()
            .reindex(weekday_order, fill_value=0)
        )
        weekday_positions = np.arange(len(weekday_labels))
        ax_weekday.bar(weekday_positions, weekday_mean.values, color="#d7a84b")
        ax_weekday.set_xticks(weekday_positions, weekday_labels)
        ax_weekday.set_title("Mittlere Nachfrage je Wochentag")
        ax_weekday.set_ylabel("Mittlere Nachfrage (kg)")

store_label = "Filiale" if store_count == 1 else "Filialen"
fig.suptitle(
    f"{store_count} {store_label}, je drei repräsentative FCM-Produkte: "
    f"erste {plot_window_size} aktive Tage",
    fontsize=14, y=1.01
)
fig.tight_layout()
figure_buffer = BytesIO()
fig.savefig(figure_buffer, format="png", dpi=120, bbox_inches="tight")
display(Image(data=figure_buffer.getvalue()))
plt.close(fig)


# Weekly analysis

Weekly rows represent active weeks. The weekly data retains daily diagnostics, so this section reports both the share of weeks with demand and the share of active days with demand inside those weeks.


In [ ]:
weekly = metrics[metrics["aggregation"] == "weekly"].copy()
weekly_overview = (weekly.groupby("dataset", observed=True).agg(series=("ARTIKEL_ID", "size"), stores=("MARKT_ID", "nunique"), products=("ARTIKEL_ID", "nunique"), active_weeks=("active_periods", "sum"), demand_weeks=("demand_periods", "sum"), zero_weeks=("zero_periods", "sum"), total_demand_kg=("total_demand", "sum"), demand_day_share=("demand_day_share", "mean")).reset_index())
weekly_overview["demand_week_share"] = weekly_overview["demand_weeks"] / weekly_overview["active_weeks"]
display(weekly_overview.style.format({"total_demand_kg":"{:,.1f}", "demand_day_share":"{:.1%}", "demand_week_share":"{:.1%}"}))


In [ ]:
display(distribution_table(weekly, ["active_periods", "demand_periods", "zero_periods", "demand_day_share", "demand_period_share"]).style.format(quantile_format))
weekly_plot = weekly.copy()
weekly_plot["Datensatz"] = weekly_plot["dataset"].astype(str).replace({"Weekly / Regular": "Wöchentlich / Regulär", "Weekly / Fcm": "Wöchentlich / FCM"})
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sns.histplot(data=weekly_plot, x="active_periods", hue="Datensatz", bins=30, element="step", stat="density", common_norm=False, ax=axes[0])
sns.histplot(data=weekly_plot, x="demand_periods", hue="Datensatz", bins=30, element="step", stat="density", common_norm=False, ax=axes[1])
axes[0].set(title="Aktive Wochen je Zeitreihe", xlabel="Anzahl aktiver Wochen", ylabel="Dichte")
axes[1].set(title="Nachfragewochen je Zeitreihe", xlabel="Anzahl der Nachfragewochen", ylabel="Dichte")
plt.tight_layout()
figure_buffer = BytesIO()
fig.savefig(figure_buffer, format="png", dpi=120, bbox_inches="tight")
display(Image(data=figure_buffer.getvalue()))
plt.close(fig)


## Weekly binned count distributions

In [ ]:
WEEK_BINS = [-1, 0, 1, 4, 8, 13, 26, 52, 104, 156, np.inf]
WEEK_LABELS = ["0", "1", "2-4", "5-8", "9-13", "14-26", "27-52", "53-104", "105-156", ">156"]
weekly_bins = binned_counts(weekly, ["active_periods", "demand_periods"], WEEK_BINS, WEEK_LABELS)
display(weekly_bins.pivot_table(index=["measure", "bin"], columns="dataset", values="share", observed=False).style.format("{:.1%}"))
for dataset, group in weekly.groupby("dataset", observed=True):
    print(dataset); display(top_store_products(group))


## Reading the results

Compare zero-period shares with demand-period shares to quantify intermittent demand within active observations. For weekly data, compare `demand_day_share` with `demand_week_share` to see how weekly aggregation hides daily zero-sales periods. The store tables rank stores by the number of products with positive demand and include the top product IDs by total demand.
